# 03 — QLoRA Fine-Tuning (Kaggle 2×T4)

**Purpose:** Orchestrate the QLoRA training job.  
All training logic lives in `src/training/train.py`; this notebook handles
Kaggle-specific setup and produces report figures.

**Two runs (both logged to W&B):**
| Run | Sampler | Goal |
|-----|---------|------|
| `qlora_uniform` | `--sampler uniform` | No shift correction — fine-tuned baseline |
| `qlora_weighted` | `--sampler weighted` | Importance-weighted — shift-corrected |

**Instructions for Kaggle:**
1. Upload this repo as a Kaggle Dataset **or** `!git clone <repo>` in the working dir.
2. Attach the IU CXR dataset: `raddar/chest-xrays-indiana-university`.
3. Enable GPU **2×T4**, enable internet (for HuggingFace Hub + W&B).
4. Set `WANDB_API_KEY` in Kaggle Secrets, or run `wandb login` in cell 4.
5. Set `HF_TOKEN` in Kaggle Secrets (required for `google/medgemma-4b-it`).
6. Run all cells in order.

**T4 optimisations applied automatically by `train.py`:**
- `fp16` (T4 native tensor cores — bf16 is software-emulated on Turing)
- `torchrun --nproc_per_node=2` when 2 GPUs detected (DDP data parallelism, not pipeline)
- `max_seq_length=512` (shorter than default — IU reports are ~150-300 tokens)
- `batch_size=2` per GPU, `grad_acc=8` → effective batch 32

**Expected runtime:** ~10–13 h total on 2×T4 (vs ~34 h without these optimisations).

## STEP 1 — Environment detection and path setup

Detect Kaggle vs Lightning AI; resolve all paths without hardcoding.

In [ ]:
import os
import sys
from pathlib import Path

ON_KAGGLE = Path('/kaggle').exists()

if ON_KAGGLE:
    # --- Kaggle layout ---
    # Clone repo if not present (e.g. if uploaded as a dataset, just point to it)
    REPO_ROOT = Path('/kaggle/working/ReportCXR')
    if not REPO_ROOT.exists():
        import subprocess
        subprocess.run(['git', 'clone', 'https://github.com/YOUR_USER/ReportCXR.git',
                        str(REPO_ROOT)], check=True)

    # Raw IU dataset is at this Kaggle input path:
    DATA_ROOT = Path('/kaggle/input/chest-xrays-indiana-university')
    IMAGES_DIR = DATA_ROOT / 'images' / 'images_normalized'

    os.chdir(REPO_ROOT)
    sys.path.insert(0, str(REPO_ROOT))
else:
    # --- Lightning AI / local layout ---
    REPO_ROOT = Path('..').resolve()   # notebook is in notebooks/
    DATA_ROOT = REPO_ROOT / 'data' / 'raw'
    IMAGES_DIR = DATA_ROOT / 'images' / 'images_normalized'

    os.chdir(REPO_ROOT)
    sys.path.insert(0, str(REPO_ROOT))

PROCESSED_DIR  = REPO_ROOT / 'data' / 'processed'
FIGURES_DIR    = REPO_ROOT / 'reports' / 'figures'
CHECKPOINT_DIR = REPO_ROOT / 'checkpoints'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f'Environment : {"Kaggle" if ON_KAGGLE else "Local/Lightning"}')
print(f'Repo root   : {REPO_ROOT}')
print(f'Images dir  : {IMAGES_DIR}  (exists={IMAGES_DIR.exists()})')
print(f'Processed   : {PROCESSED_DIR}  (exists={PROCESSED_DIR.exists()})')

## STEP 2 — Install dependencies

In [ ]:
# Install / upgrade dependencies not pre-installed on Kaggle.
# Pin lower bounds that are known-good with MedGemma + bitsandbytes on T4/CUDA 12.
!pip install -q \
    "transformers>=4.47.0" \
    "peft>=0.13.0" \
    "bitsandbytes>=0.44.0" \
    "accelerate>=0.34.0" \
    "trl>=0.12.0" \
    f1chexbert \
    scikit-multilearn \
    wandb \
    evaluate \
    bert_score \
    rouge_score \
    nltk

## STEP 3 — HuggingFace + W&B login

Set `HF_TOKEN` and `WANDB_API_KEY` via Kaggle Secrets (recommended) or paste directly.

In [ ]:
import os

# ── HuggingFace ──────────────────────────────────────────────────────────────
# Required to download google/medgemma-4b-it (gated model).
# On Kaggle: add HF_TOKEN to Secrets > Environment Variables.
hf_token = os.environ.get('HF_TOKEN', '')
if hf_token:
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
    print('HuggingFace: logged in via HF_TOKEN secret')
else:
    print('WARNING: HF_TOKEN not set — medgemma download may fail on first run')
    print('         Set it in Kaggle Secrets or run: !huggingface-cli login')

# ── W&B ──────────────────────────────────────────────────────────────────────
import wandb

wandb_key = os.environ.get('WANDB_API_KEY', '')
if wandb_key:
    wandb.login(key=wandb_key, relogin=True)
    print('W&B: logged in via WANDB_API_KEY secret')
else:
    wandb.login()  # prompts interactively
    print('W&B: logged in interactively')

## STEP 4 — Verify data and GPU

Check that processed parquet files exist, images are accessible, and GPU is available.
If processed parquets are missing, regenerate them from the raw CSVs.

In [ ]:
import logging

import torch
import yaml
import pandas as pd

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')

with open(REPO_ROOT / 'params.yaml') as f:
    params = yaml.safe_load(f)

# GPU info — detect count for DDP decision
N_GPUS = torch.cuda.device_count()
print(f'GPUs available: {N_GPUS}')
for i in range(N_GPUS):
    props = torch.cuda.get_device_properties(i)
    cc = f'{props.major}.{props.minor}'
    prec = 'bf16' if props.major >= 8 else 'fp16 (Turing native — bf16 is slow here)'
    print(f'  GPU {i}: {props.name}  {props.total_memory/1e9:.1f} GB  cc={cc}  [{prec}]')

if N_GPUS > 1:
    print(f'\nDDP enabled: torchrun --nproc_per_node={N_GPUS} → ~{N_GPUS}x throughput')
else:
    print('\nSingle GPU — training sequentially')

# Parquet checks
for split in ('train', 'val', 'test'):
    p = PROCESSED_DIR / f'{split}.parquet'
    if p.exists():
        df = pd.read_parquet(p)
        print(f'{split}.parquet : {len(df)} studies — OK')
    else:
        print(f'MISSING: {p}')

# Regenerate if missing
TRAIN_OK = (PROCESSED_DIR / 'train.parquet').exists()
if not TRAIN_OK:
    print('\nProcessed parquets missing — regenerating...')
    if ON_KAGGLE:
        os.environ['RAW_DIR_OVERRIDE'] = str(DATA_ROOT)
    !python -m src.data.load
    !python -m src.data.labels
    !python -m src.data.split
    print('Regeneration complete.')

# Image spot-check — use hasattr/__len__ instead of isinstance(x, list)
# because pyarrow reads list columns as a non-list type when loaded back from parquet
train_df = pd.read_parquet(PROCESSED_DIR / 'train.parquet')
has_frontal = train_df['frontal'].apply(lambda x: hasattr(x, '__len__') and len(x) > 0)
print(f'\nStudies with frontal image: {has_frontal.sum()}/{len(train_df)}')
if has_frontal.sum() > 0:
    sample_frontal = list(train_df[has_frontal].iloc[0]['frontal'])[0]
    img_path = IMAGES_DIR / sample_frontal
    print(f'Image spot-check: {img_path}  (exists={img_path.exists()})')
    if not img_path.exists():
        print('WARNING: images not found — check symlink / IMAGES_DIR')
else:
    print('WARNING: 0 frontal images — check src.data.load build_study_df')

## STEP 5 — Run 1: Uniform sampling (no shift correction)

Fine-tunes MedGemma with standard sequential sampling.  
This is the **fine-tuned baseline** — no distributional correction.

Logs to W&B run `qlora_uniform`. Best checkpoint saved to `checkpoints/qlora_uniform/best_model/`.

In [ ]:
import json

UNIFORM_RESULTS = CHECKPOINT_DIR / 'qlora_uniform' / 'training_results.json'

if UNIFORM_RESULTS.exists():
    with open(UNIFORM_RESULTS) as f:
        res = json.load(f)
    print(f'Uniform run already done — best epoch={res["best_epoch"]}, '
          f'best micro F1={res["best_val_f1_chexbert_micro"]:.4f}')
    print('Delete checkpoints/qlora_uniform/ to re-run.')
    RUN_UNIFORM = False
else:
    print('No cached results found — will train.')
    RUN_UNIFORM = True

In [ ]:
if RUN_UNIFORM:
    images_dir_arg = f'--images_dir {IMAGES_DIR}' if ON_KAGGLE else ''
    # Use torchrun for DDP when 2 GPUs are available (each GPU processes batches in parallel).
    # device_map="auto" (model parallelism) is replaced automatically by train.py when LOCAL_RANK is set.
    if N_GPUS > 1:
        !torchrun --nproc_per_node={N_GPUS} -m src.training.train \
            --sampler uniform \
            --run_name qlora_uniform \
            {images_dir_arg}
    else:
        !python -m src.training.train \
            --sampler uniform \
            --run_name qlora_uniform \
            {images_dir_arg}
else:
    print('Skipped (cached).')

## STEP 6 — Run 2: Importance-weighted sampling (shift correction)

Fine-tunes with `WeightedRandomSampler` using `p_target` from `params.yaml`.  
Up-weights rare-label studies so the model doesn't collapse to predicting "No Finding".

Logs to W&B run `qlora_weighted`. Best checkpoint saved to `checkpoints/qlora_weighted/best_model/`.

In [ ]:
WEIGHTED_RESULTS = CHECKPOINT_DIR / 'qlora_weighted' / 'training_results.json'

if WEIGHTED_RESULTS.exists():
    with open(WEIGHTED_RESULTS) as f:
        res = json.load(f)
    print(f'Weighted run already done — best epoch={res["best_epoch"]}, '
          f'best micro F1={res["best_val_f1_chexbert_micro"]:.4f}')
    print('Delete checkpoints/qlora_weighted/ to re-run.')
    RUN_WEIGHTED = False
else:
    print('No cached results found — will train.')
    RUN_WEIGHTED = True

In [ ]:
if RUN_WEIGHTED:
    images_dir_arg = f'--images_dir {IMAGES_DIR}' if ON_KAGGLE else ''
    if N_GPUS > 1:
        !torchrun --nproc_per_node={N_GPUS} -m src.training.train \
            --sampler weighted \
            --run_name qlora_weighted \
            {images_dir_arg}
    else:
        !python -m src.training.train \
            --sampler weighted \
            --run_name qlora_weighted \
            {images_dir_arg}
else:
    print('Skipped (cached).')

## STEP 7 — Load training results from both runs

In [ ]:
import json
import numpy as np
import pandas as pd

results = {}
for variant in ('uniform', 'weighted'):
    rpath = CHECKPOINT_DIR / f'qlora_{variant}' / 'training_results.json'
    if rpath.exists():
        with open(rpath) as f:
            results[variant] = json.load(f)
        print(f'[{variant}] best_epoch={results[variant]["best_epoch"]}  '
              f'best_micro_F1={results[variant]["best_val_f1_chexbert_micro"]:.4f}')
    else:
        print(f'[{variant}] results not found — run STEP 5/6 first')

# Also load baseline results for comparison
baseline_path = REPO_ROOT / 'reports' / 'baseline_results.json'
baseline_results = json.loads(baseline_path.read_text()) if baseline_path.exists() else None
if baseline_results:
    m = baseline_results['metrics']
    print(f'[zero-shot] micro F1={m["f1_chexbert_micro_present"]:.4f}  '
          f'macro F1={m["f1_chexbert_macro_present"]:.4f}')
else:
    print('[zero-shot] baseline_results.json not found — run notebook 02 first')

## STEP 8 — Figure: Training loss comparison

Overlay loss curves for both sampler variants on the same axes.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['figure.dpi'] = 120

fig, ax = plt.subplots(figsize=(11, 4))

colors = {'uniform': '#1f77b4', 'weighted': '#d62728'}
for variant, color in colors.items():
    if variant not in results:
        continue
    log = results[variant].get('log_history', [])
    steps  = [e['step']  for e in log if 'loss' in e]
    losses = [e['loss']  for e in log if 'loss' in e]
    if steps:
        ax.plot(steps, losses, color=color, linewidth=1.5, label=f'sampler={variant}', alpha=0.9)

ax.set_xlabel('Training step')
ax.set_ylabel('Loss (causal LM)')
ax.set_title('QLoRA Training Loss — Uniform vs Weighted Sampler')
ax.legend(fontsize=11)
plt.tight_layout()

fig_path = FIGURES_DIR / 'train_loss_comparison.png'
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'Saved {fig_path}')
plt.show()

## STEP 9 — Figure: Val F1-CheXbert per epoch

Micro and macro F1 over epochs for both runs, with zero-shot baseline reference lines.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

metric_names = ['val_f1_chexbert_micro', 'val_f1_chexbert_macro']
titles       = ['Micro F1-CheXbert', 'Macro F1-CheXbert']
zs_values    = [None, None]
if baseline_results:
    m = baseline_results['metrics']
    zs_values = [m['f1_chexbert_micro_present'], m['f1_chexbert_macro_present']]

colors_v = {'uniform': '#1f77b4', 'weighted': '#d62728'}
markers  = {'uniform': 'o', 'weighted': 's'}

for ax, metric, title, zs_val in zip(axes, metric_names, titles, zs_values):
    for variant in ('uniform', 'weighted'):
        if variant not in results:
            continue
        hist = results[variant].get('history', [])
        epochs = [r['epoch'] for r in hist]
        vals   = [r[metric]  for r in hist]
        ax.plot(epochs, vals, marker=markers[variant], color=colors_v[variant],
                linewidth=2, label=f'{variant}', markersize=8)

    if zs_val is not None:
        ax.axhline(zs_val, color='gray', linestyle='--', linewidth=1.2,
                   label=f'zero-shot = {zs_val:.3f}')

    ax.set_xlabel('Epoch')
    ax.set_ylabel('F1')
    ax.set_title(title)
    ax.set_ylim(0, 1.0)
    ax.legend(fontsize=10)

fig.suptitle('Val F1-CheXbert per Epoch — QLoRA Fine-Tuning', fontsize=13)
plt.tight_layout()

fig_path = FIGURES_DIR / 'train_val_f1_comparison.png'
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'Saved {fig_path}')
plt.show()

## STEP 10 — Figure: Sampler weight distribution

Shows how the importance weighting shifts sampling mass toward rare-label studies.

In [ ]:
from src.data.labels import CHEXBERT_LABELS
from src.training.sampler import build_sample_weights

train_df = pd.read_parquet(PROCESSED_DIR / 'train.parquet')
train_df = train_df[train_df['findings'].notna() & (train_df['findings'].str.strip() != '')]
label_matrix = train_df[CHEXBERT_LABELS].values.astype(float)

p_target   = params['sampler'].get('p_target', {})
weight_clip = float(params['sampler'].get('weight_clip', 10.0))

weights_uniform  = build_sample_weights(label_matrix, p_target=None, weight_clip=weight_clip)
weights_weighted = build_sample_weights(label_matrix, p_target=p_target, weight_clip=weight_clip)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, weights, title, color in [
    (axes[0], weights_uniform,  'Uniform sampler weights', '#1f77b4'),
    (axes[1], weights_weighted, 'Importance weights (weighted sampler)', '#d62728'),
]:
    ax.hist(weights, bins=60, color=color, alpha=0.8, edgecolor='white', linewidth=0.3)
    ax.axvline(weights.mean(), color='black', linestyle='--', linewidth=1.5,
               label=f'mean={weights.mean():.2f}')
    ax.axvline(np.median(weights), color='red', linestyle=':', linewidth=1.5,
               label=f'median={np.median(weights):.2f}')
    ax.set_xlabel('Sample weight')
    ax.set_ylabel('Training studies')
    ax.set_title(title)
    ax.legend(fontsize=10)

# Annotate ESS for weighted sampler
ess = weights_weighted.sum()**2 / (weights_weighted**2).sum()
n   = len(weights_weighted)
axes[1].set_title(
    f'Importance weights — ESS={ess:.0f}/{n} ({100*ess/n:.1f}%)'
)

plt.tight_layout()
fig_path = FIGURES_DIR / 'train_sampler_weights_comparison.png'
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'Saved {fig_path}')
plt.show()

## STEP 11 — Figure: Per-label F1 comparison

Side-by-side per-label F1 for zero-shot vs uniform fine-tune vs weighted fine-tune,
at the best checkpoint of each run.

In [ ]:
# Collect per-label F1 from each run's best epoch
def _best_per_label(results_dict: dict) -> dict:
    """Return per_label_f1 dict from the best epoch in history."""
    hist = results_dict.get('history', [])
    if not hist:
        return {}
    best = max(hist, key=lambda r: r['val_f1_chexbert_micro'])
    return best.get('per_label_f1', {})

series = {}
if baseline_results:
    series['zero-shot'] = pd.Series(baseline_results['metrics']['per_label_f1_present'])
for variant in ('uniform', 'weighted'):
    if variant in results:
        pl = _best_per_label(results[variant])
        if pl:
            series[variant] = pd.Series(pl)

if not series:
    print('No results to plot — run training first.')
else:
    df_plot = pd.DataFrame(series).reindex(CHEXBERT_LABELS).fillna(0)

    x = np.arange(len(CHEXBERT_LABELS))
    n_cols = len(df_plot.columns)
    w = 0.8 / n_cols

    col_colors = {'zero-shot': '#aec7e8', 'uniform': '#1f77b4', 'weighted': '#d62728'}

    fig, ax = plt.subplots(figsize=(14, 6))
    for i, col in enumerate(df_plot.columns):
        offset = (i - n_cols / 2 + 0.5) * w
        ax.bar(x + offset, df_plot[col].values, w,
               label=col, color=col_colors.get(col, f'C{i}'), alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(CHEXBERT_LABELS, rotation=40, ha='right', fontsize=9)
    ax.set_ylabel('F1-CheXbert')
    ax.set_ylim(0, 1.0)
    ax.set_title('Per-Label F1-CheXbert: Zero-Shot vs Fine-Tuned (Uniform vs Weighted)')
    ax.legend(fontsize=11)
    plt.tight_layout()

    fig_path = FIGURES_DIR / 'train_per_label_f1_comparison.png'
    fig.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f'Saved {fig_path}')
    plt.show()

## STEP 12 — Metric summary table

Compare all three conditions: zero-shot / uniform fine-tune / weighted fine-tune.
Only clinical metrics shown here (NLG metrics are in notebook 04).

In [ ]:
rows = []

if baseline_results:
    m = baseline_results['metrics']
    rows.append({
        'Condition'        : 'Zero-shot (no fine-tune)',
        'Sampler'          : '—',
        'Best epoch'       : '—',
        'Micro F1 (present)': round(m['f1_chexbert_micro_present'], 4),
        'Macro F1 (present)': round(m['f1_chexbert_macro_present'], 4),
        'BERTScore F1'     : round(m.get('bertscore_f1', float('nan')), 4),
    })

for variant in ('uniform', 'weighted'):
    if variant not in results:
        continue
    r = results[variant]
    best_hist = max(r.get('history', [{}]), key=lambda x: x.get('val_f1_chexbert_micro', -1))
    rows.append({
        'Condition'        : f'QLoRA fine-tune',
        'Sampler'          : variant,
        'Best epoch'       : r.get('best_epoch', '—'),
        'Micro F1 (present)': round(best_hist.get('val_f1_chexbert_micro', float('nan')), 4),
        'Macro F1 (present)': round(best_hist.get('val_f1_chexbert_macro', float('nan')), 4),
        'BERTScore F1'     : float('nan'),  # computed in notebook 04
    })

if rows:
    summary_df = pd.DataFrame(rows).set_index(['Condition', 'Sampler'])
    display(summary_df)

    # Save as figure
    fig, ax = plt.subplots(figsize=(12, 1.5 + 0.6 * len(rows)))
    ax.axis('off')
    summary_df_reset = summary_df.reset_index()
    tbl = ax.table(
        cellText=summary_df_reset.values,
        colLabels=summary_df_reset.columns.tolist(),
        cellLoc='center', loc='center',
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(10)
    tbl.scale(1.0, 1.8)
    for j in range(len(summary_df_reset.columns)):
        tbl[(0, j)].set_facecolor('#2c5f8a')
        tbl[(0, j)].set_text_props(color='white', fontweight='bold')
    ax.set_title('QLoRA Training — Metric Summary (val set)', fontsize=12, pad=12)
    plt.tight_layout()

    fig_path = FIGURES_DIR / 'train_metric_summary.png'
    fig.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f'Saved {fig_path}')
    plt.show()
else:
    print('No results to display — run training first.')

## Done

Best checkpoints are in:
```
checkpoints/qlora_uniform/best_model/
checkpoints/qlora_weighted/best_model/
```

Figures saved to `reports/figures/`:

| Figure | Content |
|--------|---------|
| `train_loss_comparison.png` | Training loss curves |
| `train_val_f1_comparison.png` | Val F1-CheXbert per epoch |
| `train_sampler_weights_comparison.png` | Uniform vs importance weight distribution |
| `train_per_label_f1_comparison.png` | Per-label F1: zero-shot vs both fine-tuned variants |
| `train_metric_summary.png` | Summary table |
| `train_loss_{variant}.png` | Per-run loss (also generated by train.py) |
| `train_val_f1_{variant}.png` | Per-run F1 (also generated by train.py) |
| `train_sampler_weights.png` | Weight histogram (generated by train.py) |

**Next step:** `notebooks/04_eval_and_figures.ipynb` — full metric stack on test set,
shift experiment, calibration analysis.